## Create order-order benchmark dataset
Import list of manually-curated, high-quality domain-domain (order-order) interactions from Geist et al., 2024. Extract data and structure files from ProtCID data dump which pertain to the Pfam binding pairs in the high-quality interaction list. Save the structure files and dataset as the order-order benchmark dataset.

In [1]:
import numpy as np
import pandas as pd
import gzip, os, re, itertools
import tarfile

#Import DDI table
ddi = pd.read_csv('/Volumes/imb-luckgr/projects/interface_clustering/datasets/benchmarking/Geist-et-al_manually-curated_DDIs.csv')

In [2]:
# Filter for 'approved' types: 48 of these
approved = ddi.query("DDI_approved == 1")
print(approved.shape)

# Create text list of Pfam-Pfam name pairs of these approved types
# Formatting follows ProtCID conventions
approved_names = []
for i,r in approved.iterrows():
    names = [r['DomainName1'],r['DomainName2']]
    names_sorted = sorted(names, key=str.lower)
    approved_names.append(f'({names_sorted[0]})({names_sorted[1]})')
print(approved_names)

(48, 62)
['(eIF-5_eIF-2B)(GTP_EFTU)', '(Asp)(Pepsin-I3)', '(fn3)(Lectin_C)', '(Trypsin)(WAP)', '(ATP-synt_C)(Renin_r)', '(14-3-3)(PBP)', '(Biotin_carb_N)(PYC_OADA)', '(PI3_PI4_kinase)(SH2)', '(Arm)(Hormone_recep)', '(Peptidase_M16)(UCR_14kD)', '(PX)(Vps26)', '(ASC)(Toxin_TOLIP)', '(FAD_binding_2)(Fer2_3)', '(TbpB_B_D)(Transferrin)', '(GDNF)(RET_CLD3)', '(IL15)(IL2RB_N1)', '(PCNA_C)(XPG_N)', '(MH2)(SARA)', '(Mnd1)(TBPIP)', '(Gtr1_RagA)(LAMTOR)', '(ATP-synt_G)(Mt_ATP-synt_B)', '(AdenylateSensor)(AMPKBI)', '(Rad21_Rec8)(SMC_N)', '(Nsp1_C)(Nup54)', '(RNA_pol_Rpc34)(Sin_N)', '(NDUF_B7)(NDUF_B8)', '(Nop52)(Ribosomal_L28e)', '(Alpha-2-MRAP_N)(Ldl_recept_a)', '(ACP)(Mal_decarbox_Al)', '(MFAP1)(PRO8NT)', '(Crl)(Sigma70_r1_2)', '(Skp1_POZ)(SOCS_box)', '(AAA_2)(Proteasome)', '(SPT16)(SSrecog)', '(MRP-L27)(MRP-L47)', '(Armet)(HSP70)', '(EIF4E-T)(Helicase_C)', '(DUF3449)(DUF382)', '(CAS_C)(RasGEF)', '(STAT1_TAZ2bind)(zf-TAZ)', '(CLASP_N)(Tubulin)', '(Ank_2)(LIM)', '(EF-hand_5)(SOUL)', '(Prok-RING_4

In [ ]:
## Extraction of desired files from tar archives
# ProtCID cluster data
#filepath = '/Volumes/imb-luckgr/projects/interface_clustering/datasets/benchmarking/ProtcidData/'
#os.mkdir(filepath)
#tar = tarfile.open('/Volumes/stromjoe/projects/interfaceClustering/datasets/benchmarking/DDI/Pfam-Pfam_DomainClusterData.tar')
#for member in tar.getmembers():
    #for pair in approved_names:
        #if pair in member.name:
            #tar.extract(member, path=filepath)
#print(os.listdir(filepath))
# Files were deposited into a folder within the designated filepath lableled 'DomainClusterData' according to the member path within the tarfile archive, was faster to manually move them into the parent folder - if re-running in the future, will need to repeat this manual step or add some code to accomplish the same task (or rename filepaths below)

# ProtCID coordinate (structure) files - needs multiple levels of extraction
#filepath = '/Volumes/imb-luckgr/projects/interface_clustering/datasets/benchmarking/ProtcidInterfaces/'
#os.mkdir(filepath)
#tar = tarfile.open('/Volumes/stromjoe/projects/interfaceClustering/datasets/benchmarking/DDI/Pfam-Pfam_DomainClusterInterfaces.tar')
#for member in tar.getmembers():
    #for pair in approved_names:
        #if pair in member.name:
            #tar.extract(member, path=filepath)
#print(os.listdir(filepath))

['DomainClusterData']
['(AAA_2)(Proteasome)_338.tar', '(ACP)(Mal_decarbox_Al)_24217.tar', '(AdenylateSensor)(AMPKBI)_1509.tar', '(Ank_2)(LIM)_1820.tar', '(Armet)(HSP70)_28698.tar', '(ATP-synt_C)(Renin_r)_35981.tar', '(ATP-synt_G)(Mt_ATP-synt_B)_43303.tar', '(Biotin_carb_N)(PYC_OADA)_2903.tar', '(CLASP_N)(Tubulin)_31259.tar', '(Crl)(Sigma70_r1_2)_34981.tar', '(DUF3449)(DUF382)_23279.tar', '(eIF-5_eIF-2B)(GTP_EFTU)_18330.tar', '(FAD_binding_2)(Fer2_3)_7585.tar', '(GDNF)(RET_CLD3)_31870.tar', '(Gtr1_RagA)(LAMTOR)_24384.tar', '(Importin_rep)(Ras)_29780.tar', '(KH_6)(MPP6)_24079.tar', '(MH2)(SARA)_11741.tar', '(MRP-63)(Ribosomal_L30)_11865.tar', '(MRP-L27)(MRP-L47)_11871.tar', '(NDUF_B7)(NDUF_B8)_20402.tar', '(Nop52)(Ribosomal_L28e)_24520.tar', '(Nsp1_C)(Nup54)_12875.tar', '(Peptidase_M16)(UCR_14kD)_14082.tar', '(PH_16)(Ras)_25599.tar', '(PI3_PI4_kinase)(SH2)_13517.tar', '(Prok-RING_4)(UQ_con)_14690.tar', '(PX)(Vps26)_19904.tar', '(Rad21_Rec8)(SMC_N)_15402.tar', '(RNA_pol_Rpc34)(Sin_N)_2475

In [ ]:
# Subsequent levels of extraction
#filepath = '/Volumes/imb-luckgr/projects/interface_clustering/datasets/benchmarking/ProtcidInterfaces/'
#for file in os.listdir(filepath):
    #tar = tarfile.open(os.path.join(filepath,file))
    #outputdir = os.path.join(filepath,file.split('.')[0])
    #print(outputdir)
    #os.mkdir(outputdir)
    #tar.extractall(path=outputdir)
    #tar.close()
    #os.remove(os.path.join(filepath,file))

#for folder in os.listdir(filepath):
    #for file in os.listdir(os.path.join(filepath,folder)):
        #tar = tarfile.open(os.path.join(filepath,folder,file))
        #outputdir = os.path.join(filepath,folder,file.split('.')[0])
        #print(outputdir)
        #os.mkdir(outputdir)
        #tar.extractall(path=outputdir)
        #tar.close()
        #os.remove(os.path.join(filepath,folder,file))

/mnt/c/Users/stromjoe/Documents/projects/interfaceClustering/datasets/benchmarking/protcid/ProtcidInterfaces/(AAA_2)(Proteasome)_338
/mnt/c/Users/stromjoe/Documents/projects/interfaceClustering/datasets/benchmarking/protcid/ProtcidInterfaces/(ACP)(Mal_decarbox_Al)_24217
/mnt/c/Users/stromjoe/Documents/projects/interfaceClustering/datasets/benchmarking/protcid/ProtcidInterfaces/(AdenylateSensor)(AMPKBI)_1509
/mnt/c/Users/stromjoe/Documents/projects/interfaceClustering/datasets/benchmarking/protcid/ProtcidInterfaces/(Ank_2)(LIM)_1820
/mnt/c/Users/stromjoe/Documents/projects/interfaceClustering/datasets/benchmarking/protcid/ProtcidInterfaces/(Armet)(HSP70)_28698
/mnt/c/Users/stromjoe/Documents/projects/interfaceClustering/datasets/benchmarking/protcid/ProtcidInterfaces/(ATP-synt_C)(Renin_r)_35981
/mnt/c/Users/stromjoe/Documents/projects/interfaceClustering/datasets/benchmarking/protcid/ProtcidInterfaces/(ATP-synt_G)(Mt_ATP-synt_B)_43303
/mnt/c/Users/stromjoe/Documents/projects/interfaceCl

In [3]:
# Search ProtCID cluster data and retrieve all data for the approved DDI types
cluster_data = pd.DataFrame({})
for file in os.listdir('/Volumes/imb-luckgr/projects/interface_clustering/datasets/benchmarking/ProtcidData/DomainClusterData'):
    filepath = os.path.join('/Volumes/imb-luckgr/projects/interface_clustering/datasets/benchmarking/ProtcidData/DomainClusterData/',file)
    with gzip.open(filepath, mode='rt') as f:
        tmpdf = pd.read_csv(f, delimiter='\t', index_col=False)
        cluster_data = pd.concat([cluster_data,tmpdf], axis=0)

# Formatting of Pfam-Pfam pair identifiers
encodena = []
for x in cluster_data['Relation']:
    try:
        int(x)
    except:
        encodena.append(x)
    else:
        encodena.append(np.nan)
cluster_data.insert(0,'pairID',encodena)
cluster_data['pairID'] = cluster_data['pairID'].ffill()
cluster_data = cluster_data.dropna(subset=['CFID'])
cluster_data['pairID'] = [''.join(x.split(';')) for x in cluster_data['pairID']]
cluster_data['DomainInterfaceID'] = [int(x) for x in cluster_data['DomainInterfaceID']]
cluster_data.reset_index(drop=True, inplace=True)
print(cluster_data)


                      pairID Relation  ClusterID  CFID SpaceGroup CrystForm  \
0        (AAA_2)(Proteasome)      338          1   2.0  P 21 21 2    A12B12   
1        (AAA_2)(Proteasome)      338          1   2.0  P 21 21 2    A12B12   
2        (AAA_2)(Proteasome)      338          1   2.0  P 21 21 2    A12B12   
3        (AAA_2)(Proteasome)      338          1   2.0  P 21 21 2    A12B12   
4        (AAA_2)(Proteasome)      338          1   2.0  P 21 21 2    A12B12   
..                       ...      ...        ...   ...        ...       ...   
977  (TbpB_B_D)(Transferrin)    11538          1   3.0  P 43 21 2        AB   
978           (Trypsin)(WAP)    17709          1   1.0   P 1 21 1        AB   
979           (Trypsin)(WAP)    17709          1   2.0   P 1 21 1      A3B2   
980           (Trypsin)(WAP)    17709          1   2.0   P 1 21 1      A3B2   
981           (Trypsin)(WAP)    17709          1   3.0   P 4 21 2        AB   

    PdbID  DomainInterfaceID  SurfaceArea Interface

In [4]:
# Filter for approved DDI types
approved_clusters = cluster_data.query("pairID in @approved_names")
approved_clusters['IFID'] = ['IF'+str(x) for x in approved_clusters.index]

# Search for appropriate PDB file within ProtCID interface coordinate folders, extract original author chain IDs, and save in new folder

parentpath = '/Volumes/imb-luckgr/projects/interface_clustering/datasets/benchmarking/ProtcidInterfaces/'
chaina = pd.Series(list(itertools.repeat(np.nan, approved_clusters.shape[0])), index=approved_clusters.index)
chainb = pd.Series(list(itertools.repeat(np.nan, approved_clusters.shape[0])), index=approved_clusters.index)
for i,r in approved_clusters.iterrows():
    pairpath = os.path.join(parentpath,"_".join([r['pairID'],r['Relation']]))
    clusterpath = os.path.join(pairpath, "_".join([r['pairID'],r['Relation'],str(r['ClusterID'])]))
    success = False
    for filename in os.listdir(clusterpath):
        if filename == r['PdbID']+'_d'+str(r['DomainInterfaceID'])+'.pdb':
            success=True
            with open(os.path.join(clusterpath,filename),'r') as f:
                content = f.readlines()
                # Some PDB files do not follow the same structure - these appear to be intrachain DDIs, which we are not interested in right now. It is sufficient to ignore these
            chaina[i] = re.findall('(Author Chain)\s(\S+)\s', content[1])[0][1]
            chainb[i] = re.findall('(Author Chain)\s(\S+)\s', content[2])[0][1]
            with open(f"/Volumes/imb-luckgr/projects/interface_clustering/datasets/benchmarking/DDI/{r['PdbID']}-{r['IFID']}.pdb",'w') as g:
                    g.writelines(content)
                
print(chaina, '\n', chainb)

<>:20: SyntaxWarning: invalid escape sequence '\s'
<>:21: SyntaxWarning: invalid escape sequence '\s'
<>:20: SyntaxWarning: invalid escape sequence '\s'
<>:21: SyntaxWarning: invalid escape sequence '\s'
/var/folders/8n/b4ym5rbn48v7d2lxw_5ph7z40000gp/T/ipykernel_51462/2121366560.py:20: SyntaxWarning: invalid escape sequence '\s'
  chaina[i] = re.findall('(Author Chain)\s(\S+)\s', content[1])[0][1]
/var/folders/8n/b4ym5rbn48v7d2lxw_5ph7z40000gp/T/ipykernel_51462/2121366560.py:21: SyntaxWarning: invalid escape sequence '\s'
  chainb[i] = re.findall('(Author Chain)\s(\S+)\s', content[2])[0][1]
/var/folders/8n/b4ym5rbn48v7d2lxw_5ph7z40000gp/T/ipykernel_51462/2121366560.py:20: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'A' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  chaina[i] = re.findall('(Author Chain)\s(\S+)\s', content[1])[0][1]
/var/folders/8n/b4ym5rbn48v7

0        A
1        A
2      NaN
3      NaN
4      NaN
      ... 
977      A
978      I
979      A
980    NaN
981      E
Length: 982, dtype: object 
 0        G
1        G
2      NaN
3      NaN
4      NaN
      ... 
977      B
978      E
979      B
980    NaN
981      I
Length: 982, dtype: object


In [5]:
# Append to dataframe and drop any rows without chain assignment
approved_clusters['chainA'] = chaina
approved_clusters['chainB'] = chainb
approved_clusters = approved_clusters.dropna(subset=['chainA','chainB'])
approved_clusters['IFID'] = ['IF'+str(x) for x in approved_clusters.index]
print(approved_clusters.shape[0])

660


In [6]:
# Remove structures from the benchmark dataset that did not get aligned in the end due to inability to properly process PDB file
too_short = ['IF22','IF24','IF27','IF13','IF19','IF290']
approved_clusters = approved_clusters[~approved_clusters["IFID"].isin(too_short)]

# Assign cluster numbers
approved_clusters['ClusterID'] = [str(x) for x in approved_clusters.ClusterID]
approved_clusters.sort_values(by=["pairID","ClusterID"], inplace=True) #This order will be used to standardize heatmap visuals
approved_clusters['trueClusters'] = pd.factorize(approved_clusters.pairID + approved_clusters.ClusterID)[0]
approved_clusters['full_clusterID'] = approved_clusters.pairID + approved_clusters.ClusterID #Pfam pair plus ProtCID cluster number
approved_clusters['fullID'] = approved_clusters[['PdbID', 'IFID']].agg('-'.join, axis=1) #Identifier for each individual interface
# Write final dataframe to disk
approved_clusters.to_csv('/Volumes/imb-luckgr/projects/interface_clustering/datasets/benchmarking/ddi_clusters.csv',
                         index=False)

In [7]:
# Write TSV file containing full interface IDs and ProtCID cluster IDs for reference in Foldseek regression test
with open('/Users/stromjoe/Documents/ddi_lookup_bench.tsv', 'w', newline='\n') as f:
    for i in approved_clusters.index:
        f.write(approved_clusters.loc[i,'fullID']+"\t"+approved_clusters.loc[i,'full_clusterID']+"\n")

In [8]:
# Parse PDB files in ProtCID folder and write them back out because iAlign's parsing module is not robust
from Bio import PDB
parser = PDB.PDBParser(QUIET=True)
io = PDB.PDBIO()
filepath = '/Volumes/imb-luckgr/projects/interface_clustering/datasets/benchmarking/DDI/'
for file in os.listdir(filepath):
    if len(file.split(".")) == 2 and (file.split(".")[1] == 'pdb'):
        filename = os.path.join(filepath, file)
        name = file.split(".")[0]
        structure = parser.get_structure(name, filename)
        io.set_structure(structure)
        io.save(filename)

In [9]:
# Cluster statistics
approved_clusters = pd.read_csv("/Volumes/imb-luckgr/projects/interface_clustering/datasets/benchmarking/ddi_clusters.csv")
# Approved DDI types with no cluster data in ProtCID
print(list(set(approved_names) - set(approved_clusters.pairID.unique())))
# Number of structures from ProtCID clusters
print('Number of structures from ProtCID clusters: ', approved_clusters.shape[0])
# Statistics for number of clusters per DDI type
print('Statistics for number of clusters for each DDI type:\n', 
      approved_clusters.groupby('pairID')['ClusterID'].max().describe())
# Statistics for number of structures per cluster
print('Statistics for number of structures per cluster:\n',
      approved_clusters.groupby(['pairID','ClusterID'])['IFID'].count().describe())
# List of DDI types and clusters
print(approved_clusters.groupby(['pairID','ClusterID'])['IFID'].count())

['(Alpha-2-MRAP_N)(Ldl_recept_a)', '(EIF4E-T)(Helicase_C)', '(CAS_C)(RasGEF)', '(EF-hand_5)(SOUL)', '(MFAP1)(PRO8NT)', '(IL15)(IL2RB_N1)', '(Armet)(HSP70)', '(DUF3449)(DUF382)', '(Mnd1)(TBPIP)', '(14-3-3)(PBP)', '(STAT1_TAZ2bind)(zf-TAZ)', '(Asp)(Pepsin-I3)', '(Rad21_Rec8)(SMC_N)', '(fn3)(Lectin_C)', '(RNA_pol_Rpc34)(Sin_N)', '(ASC)(Toxin_TOLIP)', '(Arm)(Hormone_recep)', '(PCNA_C)(XPG_N)']
Number of structures from ProtCID clusters:  654
Statistics for number of clusters for each DDI type:
 count    30.000000
mean      1.433333
std       0.897634
min       1.000000
25%       1.000000
50%       1.000000
75%       1.000000
max       4.000000
Name: ClusterID, dtype: float64
Statistics for number of structures per cluster:
 count    41.000000
mean     15.951220
std      23.185072
min       1.000000
25%       3.000000
50%       6.000000
75%      14.000000
max      83.000000
Name: IFID, dtype: float64
pairID                       ClusterID
(AAA_2)(Proteasome)          1             5
       